In [0]:
df = spark.sql('select * from databricksreddy.bronze.orders')

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = df.withColumn("order_date", to_date(col("order_date"), "MM-dd-yyyy"))

In [0]:
df.display()

In [0]:
df = df.withColumn('year',year(col("order_date")))
df.display()

In [0]:
class window_fun:
    def rank(self,df):
        df = df.withColumn('rank',rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df
    
    def dense_rank(self,df):
        df = df.withColumn('dense_rank',dense_rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df
    
    def row_number(self,df):
        df = df.withColumn('row_number',row_number().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df

In [0]:
window = window_fun()
df_window = window.row_number(df)

In [0]:
df_window.display()

In [0]:
df.display()

In [0]:
%sql
create schema if not exists databricksreddy.silver

In [0]:
df.write.format('delta').mode('overwrite').save('abfss://silver@datalakevishnu1.dfs.core.windows.net/orders')

In [0]:
df.write.mode('overwrite').saveAsTable('databricksreddy.silver.orders')